# Reproduce: gap-protected spectral invariants on T^d/Z_2

Run all cells top to bottom (in Colab: *Runtime -> Run all*). The next
cell prepares the environment; the one after recomputes each headline
number from first principles and lays it beside the value **printed in
the paper**, with the equation or theorem number and a match mark. The
last cell runs the full verification suite.

Paper: *Gap-protected spectral invariants on T^d/Z_2: dimensional rigidity
at d = 4* (doi:10.5281/zenodo.20597196).


In [ ]:
# Setup: no-op locally; in Colab/fresh Jupyter it fetches the repo and deps.
import os, subprocess, sys
if not os.path.exists('run_all.py'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kootru-repo/'
                    'gap-protected-spectral-invariants', '_repo'], check=True)
    os.chdir('_repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
print('ready in', os.getcwd())


In [2]:
import itertools, math
import mpmath as mp
from IPython.display import display, HTML
from collections import OrderedDict
mp.mp.dps = 30

# ---------------- computation ----------------
pts = [n for n in itertools.product(range(-3, 4), repeat=4)
       if sum(x*x for x in n) <= 5]

def kraw(h, w, d=4):
    return sum((-1)**j * math.comb(w, j) * math.comb(d - w, h - j)
               for j in range(h + 1) if 0 <= h - j <= d - w)

def N4(K):
    return sum(1 for n in itertools.product(range(-3, 4), repeat=4)
               if sum(x*x for x in n) <= K)

r4 = [N4(k) - (N4(k-1) if k else 0) for k in range(6)]
b0, chi, dyn = r4[0], r4[1], sum(r4[2:6])

g = [sum((-1)**sum(n[k] for k in range(h)) for n in pts) for h in range(5)]
lam = [int(sum(kraw(h, w) * g[h] for h in range(5))) for w in range(5)]

pi, gE, ln2 = mp.pi, mp.euler, mp.log(2)
Z = [-8*ln2, (pi - 2*ln2)/2, -2*ln2, (-pi - 2*ln2)/2, -4*ln2]
Gr = [z/(4*pi**2) for z in Z]
mu = [sum(Gr[h]*kraw(h, w) for h in range(5)) for w in range(5)]
mu_paper = [-0.562, 0.089, -0.140, -0.229, -0.281]

F0 = 1 - mp.log(pi) + 6*mp.zeta(2, 1, 1)/pi**2 + mp.log(4)/3
D1 = F0 + gE/2
action = 137 + D1
rho = max(abs(m) for m in mu)/(8*pi**2)
rem = rho**2/(1 - rho)
lo, hi = action - rem, action + rem

# ---------------- claims ----------------
rows = []
def rec(group, ref, claim, paper, comp, ok):
    rows.append((group, ref, claim, paper, comp, bool(ok)))

A = "1.  The integer at the centre:  N_4(5) = 137"
rec(A, "Jacobi 4-square; &sect;5",
    "shell counts r<sub>4</sub>(k),&nbsp; k = 0&hellip;5",
    "[1, 8, 24, 32, 24, 48],&nbsp; &Sigma; = 137",
    "%s,  sum = %d" % (r4, sum(r4)),
    r4 == [1, 8, 24, 32, 24, 48] and sum(r4) == 137)
rec(A, "Eq.&nbsp;(2), Thm&nbsp;5.1",
    "N<sub>4</sub>(5) = b<sub>0</sub> + &chi;<sub>orb</sub> + |Fix|&middot;&chi;<sub>orb</sub>",
    "1 + 8 + 128 = 137",
    "%d + %d + %d = %d" % (b0, chi, dyn, b0 + chi + dyn),
    (b0, chi, dyn) == (1, 8, 128))
rec(A, "Eq.&nbsp;(2)",
    "each summand an independent invariant",
    "&chi;<sub>orb</sub> = 8,&nbsp; |Fix|&middot;&chi;<sub>orb</sub> = 16&middot;8 = 128",
    "chi_orb = %d,  16*%d = %d" % (chi, chi, 16*chi),
    chi == 8 and 16*chi == dyn == 128)

B = "2.  Krawtchouk spectral data on the 16 fixed points"
rec(B, "&sect;6 table",
    "integer Gram eigenvalues &lambda;<sub>w</sub> &nbsp;(w = 0&hellip;4)",
    "[144, 224, 64, 128, 256] &nbsp; with g<sub>0</sub> = 137",
    "%s   (g0 = %d)" % (lam, g[0]),
    lam == [144, 224, 64, 128, 256] and g[0] == 137)
rec(B, "&sect;6, Eq.&nbsp;(3)",
    "scattering eigenvalues &mu;<sub>w</sub> &nbsp;(w = 0&hellip;4)",
    "(&minus;0.562, 0.089, &minus;0.140, &minus;0.229, &minus;0.281)",
    "(" + ", ".join(mp.nstr(m, 3) for m in mu) + ")",
    all(abs(float(m) - p) < 5e-4 for m, p in zip(mu, mu_paper)))

C = "3.  Fractional correction &rarr; smooth action &rarr; Born interval"
rec(C, "&sect;7",
    "F<sub>0</sub> = 1 &minus; ln&pi; + 6&zeta;&prime;(2)/&pi;<sup>2</sup> + (ln4)/3",
    "&minus;0.252592758&hellip;",
    mp.nstr(F0, 10),
    abs(F0 - mp.mpf("-0.252592758")) < 1e-9)
rec(C, "Eq.&nbsp;(5)",
    "&Delta;<sub>1</sub> = F<sub>0</sub> + &gamma;<sub>E</sub>/2",
    "0.0360151",
    mp.nstr(D1, 8),
    abs(D1 - mp.mpf("0.0360151")) < 5e-7)
rec(C, "Eq.&nbsp;(6)",
    "smooth spectral action&nbsp; N<sub>4</sub>(K*) + &Delta;<sub>1</sub>",
    "137.036015074&hellip;",
    mp.nstr(action, 12),
    abs(action - mp.mpf("137.036015074")) < 5e-10)
rec(C, "Eq.&nbsp;(3)",
    "spectral radius&nbsp; &rho; = max<sub>w</sub> |&mu;<sub>w</sub>| / (8&pi;<sup>2</sup>)",
    "&rho; &lt; 7.2&times;10<sup>&minus;3</sup>",
    mp.nstr(rho, 5),
    rho < mp.mpf("7.2e-3"))
rec(C, "Eq.&nbsp;(4)",
    "Born remainder&nbsp; &rho;<sup>2</sup>/(1&minus;&rho;)",
    "&lt; 5.3&times;10<sup>&minus;5</sup>",
    mp.nstr(rem, 5),
    rem < mp.mpf("5.3e-5"))
rec(C, "&sect;7",
    "Born interval&nbsp; N<sub>4</sub>(K*)+&Delta;<sub>1</sub> &plusmn; &rho;<sup>2</sup>/(1&minus;&rho;)",
    "&sub; [137.03596, 137.03607]",
    "[%s, %s]" % (mp.nstr(lo, 9), mp.nstr(hi, 9)),
    lo >= mp.mpf("137.03596") and hi <= mp.mpf("137.03607"))

# ---------------- render ----------------
n_ok = sum(1 for r in rows if r[-1])
n_tot = len(rows)
FONT = "font-family:system-ui,-apple-system,Segoe UI,sans-serif"

banner = ('<div style="%s;border:1px solid #e5e7eb;border-left:5px solid #2563eb;'
          'padding:12px 16px;border-radius:6px;background:#f8fafc;max-width:940px">'
          '<div style="font-size:17px;font-weight:700;color:#0f172a">'
          'Reproducing the paper&rsquo;s headline results</div>'
          '<div style="color:#475569;margin-top:3px;font-size:13.5px">'
          'Each value is recomputed from first principles and matched against the '
          'number printed in the paper, with its equation or theorem number. '
          'Blue = recomputed here.</div></div>') % FONT
display(HTML(banner))

def render(group, gr):
    # Self-contained light card: every cell carries explicit background and
    # color, so it stays legible under both light and dark notebook themes.
    mono = "font-family:ui-monospace,SFMono-Regular,Menlo,monospace;"
    h = ['<div style="max-width:940px;margin:14px 0;border:1px solid #d1d5db;'
         'border-radius:8px;overflow:hidden">']
    h.append('<div style="%s;font-weight:700;color:#0f172a;background:#e7ebf0;'
             'padding:9px 12px;font-size:14px">%s</div>' % (FONT, group))
    h.append('<table style="border-collapse:collapse;width:100%%;%s;'
             'font-size:13px;background:#ffffff">' % FONT)
    th = "text-align:left;padding:7px 10px;color:#ffffff;background:#1f2937;"
    h.append('<tr>'
             '<th style="%s">Paper</th>'
             '<th style="%s">Claim &amp; printed value</th>'
             '<th style="%s">Recomputed here</th>'
             '<th style="%stext-align:center">&nbsp;</th></tr>'
             % (th, th, th, th))
    for i, (ref, claim, paper, comp, ok) in enumerate(gr):
        bg = "#f8fafc" if i % 2 else "#ffffff"
        mbg = "#dcfce7" if ok else "#fee2e2"
        mk = ('<span style="color:#15803d;font-weight:800;font-size:16px">&#10003;</span>'
              if ok else
              '<span style="color:#b91c1c;font-weight:800;font-size:16px">&#10007;</span>')
        cell = ('padding:7px 10px;vertical-align:top;border-top:1px solid #e5e7eb;'
                'background:%s;') % bg
        h.append('<tr>')
        h.append('<td style="%scolor:#475569;white-space:nowrap">%s</td>'
                 % (cell, ref))
        h.append('<td style="%scolor:#0f172a">%s'
                 '<div style="%scolor:#0f172a;margin-top:3px">%s</div></td>'
                 % (cell, claim, mono, paper))
        h.append('<td style="%s%scolor:#1d4ed8">%s</td>' % (cell, mono, comp))
        h.append('<td style="padding:7px 10px;text-align:center;'
                 'border-top:1px solid #e5e7eb;background:%s">%s</td>' % (mbg, mk))
        h.append('</tr>')
    h.append('</table></div>')
    return ''.join(h)

groups = OrderedDict()
for group, ref, claim, paper, comp, ok in rows:
    groups.setdefault(group, []).append((ref, claim, paper, comp, ok))
for group, gr in groups.items():
    display(HTML(render(group, gr)))

allok = n_ok == n_tot
card = ('<div style="%s;margin-top:8px;max-width:940px;padding:12px 16px;'
        'border-radius:6px;font-size:16px;font-weight:700;color:#fff;background:%s">'
        '%s &nbsp;%d / %d headline numbers reproduced</div>') % (
        FONT, "#16a34a" if allok else "#dc2626",
        "&#10003;" if allok else "&#10007;", n_ok, n_tot)
display(HTML(card))
assert allok, "a headline number did not match the paper"


Paper,Claim & printed value,Recomputed here,
Jacobi 4-square; §5,"shell counts r4(k), k = 0…5[1, 8, 24, 32, 24, 48], Σ = 137","[1, 8, 24, 32, 24, 48], sum = 137",✓
"Eq. (2), Thm 5.1",N4(5) = b0 + χorb + |Fix|·χorb1 + 8 + 128 = 137,1 + 8 + 128 = 137,✓
Eq. (2),"each summand an independent invariantχorb = 8, |Fix|·χorb = 16·8 = 128","chi_orb = 8, 16*8 = 128",✓


Paper,Claim & printed value,Recomputed here,
§6 table,"integer Gram eigenvalues λw (w = 0…4)[144, 224, 64, 128, 256] with g0 = 137","[144, 224, 64, 128, 256] (g0 = 137)",✓
"§6, Eq. (3)","scattering eigenvalues μw (w = 0…4)(−0.562, 0.089, −0.140, −0.229, −0.281)","(-0.562, 0.0889, -0.14, -0.229, -0.281)",✓


Paper,Claim & printed value,Recomputed here,
§7,F0 = 1 − lnπ + 6ζ′(2)/π2 + (ln4)/3−0.252592758…,-0.2525927586,✓
Eq. (5),Δ1 = F0 + γE/20.0360151,0.036015074,✓
Eq. (6),smooth spectral action N4(K*) + Δ1137.036015074…,137.036015074,✓
Eq. (3),spectral radius ρ = maxw |μw| / (8π2)ρ < 7.2×10−3,0.0071158,✓
Eq. (4),Born remainder ρ2/(1−ρ)< 5.3×10−5,5.0998e-5,✓
§7,"Born interval N4(K*)+Δ1 ± ρ2/(1−ρ)⊂ [137.03596, 137.03607]","[137.035964, 137.036066]",✓


## Full verification suite

**Estimated time: about 1 minute on a multi-core machine, a couple
of minutes on a free Colab CPU (2 cores).**

Every script in the repository, the same command the continuous-
integration workflow runs (the 5 primary scripts, the adversarial
battery, the crystallographic sweeps, and the definitive value
recomputation). The scripts run in parallel. Run the cell to watch
a live progress bar that counts down the estimated time remaining;
the summary saved below is the finished result, with each step's
status, its check count, and the total time. It passes only if
every check passes.

In [3]:
import os, json, subprocess, sys
from IPython.display import display, HTML, update_display

FONT = "font-family:system-ui,-apple-system,'Segoe UI',sans-serif"

def _mmss(s):
    s = int(round(s or 0)); return "%d:%02d" % (s // 60, s % 60)

def _render(ev):  # live, styled bar (Colab/Jupyter only; GitHub strips styles)
    pct = int(ev["frac"] * 100)
    ct = ev.get("checks_total") or 0
    chk = ("%d/%d checks" % (ev["checks"], ct)) if ct else ("%d checks" % ev["checks"])
    rows = []
    for s in ev["steps"]:
        st = s["state"]
        if st == "done":
            col = "#16a34a" if s.get("ok", True) else "#dc2626"
            mark = "&#10003;" if s.get("ok", True) else "&#10007;"
            cnt = ("%d checks" % s["count"]) if s.get("count") is not None else "done"
            rows.append("<div style='color:%s'>%s&nbsp; %s &mdash; %s "
                        "<span style='color:#9ca3af'>(%s)</span></div>"
                        % (col, mark, s["label"], cnt, _mmss(s["dur"])))
        elif st == "running":
            rows.append("<div style='color:#b45309'>&#9656;&nbsp; %s &mdash; "
                        "running %s <span style='color:#9ca3af'>(~%s left)</span></div>"
                        % (s["label"], _mmss(s["elapsed"]), _mmss(s["remaining"])))
        else:
            rows.append("<div style='color:#9ca3af'>&#183;&nbsp; %s &mdash; queued</div>"
                        % s["label"])
    bar = ("<div style='background:#e5e7eb;border-radius:7px;height:20px;overflow:hidden;"
           "margin:8px 0'><div style='width:%d%%;height:100%%;background:"
           "linear-gradient(90deg,#2563eb,#06b6d4)'></div></div>" % pct)
    head = ("<b>%d%%</b> &middot; elapsed %s &middot; ETA %s &middot; %d/%d steps &middot; %s"
            % (pct, _mmss(ev["elapsed"]), _mmss(ev["eta"]),
               ev["steps_done"], ev["steps_total"], chk))
    return ('<div style="%s;max-width:940px">%s%s'
            '<div style="font-size:13px;line-height:1.55">%s</div></div>'
            % (FONT, head, bar, "".join(rows)))

def _summary(steps, ok, checks, ctot, wall, frac):
    # Style-free HTML: GitHub's notebook viewer renders this (same format as the
    # headline-checks tables above); only inline CSS is stripped, structure stays.
    W = 34; fill = int(round(frac * W))
    bar = "&#9608;" * fill + "&#9617;" * (W - fill)   # full block / light shade
    mark = "&#9989;" if ok else "&#10060;"            # white check / cross-mark
    head = "%s <b>Full verification suite: %s</b>" % (
        mark, ("all %d checks passed" % checks) if ok else "a script FAILED")
    rows = ""
    for s in steps:
        m = "&#9989;" if s.get("ok", True) else "&#10060;"
        res = ("%d checks" % s["count"]) if s.get("count") is not None else "passed"
        rows += "<tr><td>%s</td><td>%s %s</td><td>%s</td></tr>" % (
            s["label"], m, res, _mmss(s.get("dur", 0)))
    pct = int(round(frac * 100))
    cc = ("%d/%d" % (checks, ctot)) if ctot else str(checks)
    return ("<p>%s</p>"
            "<p><code>%s</code> &nbsp; %d%% &nbsp; %s checks &nbsp; %s</p>"
            "<table><thead><tr><th>Step</th><th>Result</th><th>Time</th></tr></thead>"
            "<tbody>%s</tbody></table>") % (head, bar, pct, cc, wall, rows)

env = dict(os.environ, VERIFY_PROGRESS="json")
proc = subprocess.Popen([sys.executable, "run_all.py"], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
did = "fvs-progress"
display(HTML("<i>starting the full verification suite&hellip;</i>"), display_id=did)
last_tick = None; end = None
for line in proc.stdout:
    line = line.strip()
    if not line:
        continue
    try:
        ev = json.loads(line)
    except ValueError:
        print(line); continue
    if ev.get("t") == "tick":
        last_tick = ev
        update_display(HTML(_render(ev)), display_id=did)
    elif ev.get("t") == "end":
        end = ev
proc.wait()
ok = proc.returncode == 0

steps = (last_tick or {}).get("steps", [])
checks = (end or last_tick or {}).get("checks", 0)
ctot = (end or last_tick or {}).get("checks_total", 0)
wall = _mmss((end or {}).get("wall", (last_tick or {}).get("elapsed", 0)))
frac = 1.0 if ok else float((last_tick or {}).get("frac", 1.0))
update_display(HTML(_summary(steps, ok, checks, ctot, wall, frac)), display_id=did)

if not ok and end:
    for f in end.get("failed", []):
        print("\n=== FAILED:", f["script"], "===")
        print((f.get("tail") or "")[-1500:])


Step,Result,Time
"Mode count, gap protection, decomposition",✅ 298 checks,0:01
Spectral radius and Born interval,✅ 37 checks,0:00
Krawtchouk eigenvalues and exact identities,✅ 17 checks,0:00
Crystallographic classification,✅ 53 checks,0:02
Computational cross-checks,✅ 45 checks,0:13
Substrate uniqueness over W(B_4),✅ passed,0:05
Substrate uniqueness over 227 CARAT classes,✅ passed,1:14
Adversarial / falsification suite,✅ 245 checks,1:16
Definitive value verification,✅ 131 checks,1:18
